In [1]:
# Install all wheels from the input notebook's package directory
!pip install --no-index --find-links=/kaggle/input/notebooks/harshankmatkar/dependencies/packages tinker-cookbook


Looking in links: /kaggle/input/notebooks/harshankmatkar/dependencies/packages
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/tinker_cookbook-0.3.1.dev13+g8fc8c1758-py3-none-any.whl
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/chz-0.4.0-py3-none-any.whl (from tinker-cookbook)
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/tinker-0.18.0-py3-none-any.whl (from tinker-cookbook)


In [2]:
import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = 32

def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    # Delta = B @ A, shape [out_dim, in_dim]
    delta = B.float() @ A_mat.float()

    # Best rank-k approximation in Frobenius norm
    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    U = U[:, :rank]
    S = S[:rank]
    Vh = Vh[:rank, :]

    sroot = torch.sqrt(S)
    B_new = U * sroot.unsqueeze(0)          # [out_dim, rank]
    A_new = sroot.unsqueeze(1) * Vh         # [rank, in_dim]

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()


def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []   # (row_start, row_end, rank)
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )
        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device
    )

    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    # Force fused modules back down to rank 32
    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK
        )
        final_rank = FORCED_FUSED_RANK

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


# monkey-patch
A._merge_fused_projections = patched_merge_fused_projections
print("patched:", A._merge_fused_projections.__name__)

patched: patched_merge_fused_projections


In [ ]:
import os
import shutil
from safetensors.torch import load_file, save_file

# ── LASER hyper-parameters ────────────────────────────────────────────────────
LASER_ENERGY_FRACTION = 0.99  # retain singular values covering this fraction of energy
LASER_MIN_RANK = 4
LASER_MAX_RANK = 32           # matches FORCED_FUSED_RANK cap above


def laser_prune_lora_pair(
    B: torch.Tensor,
    A_mat: torch.Tensor,
    energy_fraction: float = LASER_ENERGY_FRACTION,
    min_rank: int = LASER_MIN_RANK,
    max_rank: int = LASER_MAX_RANK,
):
    """Prune a LoRA (B, A) pair via truncated SVD.

    Retains the fewest singular values whose cumulative squared-energy
    covers at least `energy_fraction` of the total, clamped to
    [min_rank, max_rank].
    """
    delta = B.float() @ A_mat.float()          # [out_dim, in_dim]
    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)

    total_energy = (S ** 2).sum()
    cumulative = (S ** 2).cumsum(0)
    # first index where cumulative energy >= threshold
    k = int((cumulative < energy_fraction * total_energy).sum().item()) + 1
    k = max(min_rank, min(k, max_rank, S.shape[0]))

    sroot = torch.sqrt(S[:k])
    B_new = (U[:, :k] * sroot.unsqueeze(0)).to(B.dtype).contiguous()
    A_new = (sroot.unsqueeze(1) * Vh[:k, :]).to(A_mat.dtype).contiguous()
    return B_new, A_new


def apply_laser_to_adapter(
    src_adapter_path: str,
    dst_adapter_path: str,
    energy_fraction: float = LASER_ENERGY_FRACTION,
    min_rank: int = LASER_MIN_RANK,
    max_rank: int = LASER_MAX_RANK,
) -> str:
    """Copy adapter to `dst_adapter_path`, then apply LASER pruning in-place.

    Returns the destination path for chaining.
    """
    if os.path.exists(dst_adapter_path):
        shutil.rmtree(dst_adapter_path)
    shutil.copytree(src_adapter_path, dst_adapter_path)

    for fname in sorted(os.listdir(dst_adapter_path)):
        if not fname.endswith(".safetensors"):
            continue
        fpath = os.path.join(dst_adapter_path, fname)
        state_dict = load_file(fpath)

        lora_b_keys = [k for k in state_dict if "lora_B" in k and k.endswith(".weight")]
        n_pruned = 0
        for b_key in lora_b_keys:
            a_key = b_key.replace("lora_B", "lora_A")
            if a_key not in state_dict:
                continue
            B_new, A_new = laser_prune_lora_pair(
                state_dict[b_key], state_dict[a_key],
                energy_fraction=energy_fraction,
                min_rank=min_rank,
                max_rank=max_rank,
            )
            state_dict[b_key] = B_new
            state_dict[a_key] = A_new
            n_pruned += 1

        save_file(state_dict, fpath)
        print(f"  LASER: pruned {n_pruned} pairs in {fname}")

    return dst_adapter_path


# ── Run LASER ─────────────────────────────────────────────────────────────────
_BASE_ADAPTER_PATH = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
_LASER_ADAPTER_PATH = "/kaggle/working/adapter-laser"

adapter_path = apply_laser_to_adapter(_BASE_ADAPTER_PATH, _LASER_ADAPTER_PATH)
print(f"LASER done → adapter_path = {adapter_path}")


In [ ]:
import math

# ── SLERP ─────────────────────────────────────────────────────────────────────

def slerp(w0: torch.Tensor, w1: torch.Tensor, t: float = 0.5) -> torch.Tensor:
    """Spherical linear interpolation between two weight tensors.

    Falls back to linear interpolation when the tensors are nearly parallel
    or either norm is close to zero.
    """
    orig_shape = w0.shape
    v0 = w0.float().flatten()
    v1 = w1.float().flatten()

    norm0, norm1 = v0.norm(), v1.norm()
    if norm0 < 1e-8 or norm1 < 1e-8:
        return ((1 - t) * v0 + t * v1).reshape(orig_shape).to(w0.dtype)

    v0_n, v1_n = v0 / norm0, v1 / norm1
    dot = torch.clamp((v0_n * v1_n).sum(), -1.0, 1.0)
    omega = torch.acos(dot)
    sin_omega = torch.sin(omega)

    if sin_omega.abs() < 1e-6:               # nearly parallel → linear fallback
        return ((1 - t) * v0 + t * v1).reshape(orig_shape).to(w0.dtype)

    scale0 = torch.sin((1 - t) * omega) / sin_omega
    scale1 = torch.sin(t * omega) / sin_omega
    return (scale0 * v0 + scale1 * v1).reshape(orig_shape).to(w0.dtype)


def slerp_merge_adapters(sd0: dict, sd1: dict, t: float = 0.5) -> dict:
    """Merge two adapter state dicts via per-tensor SLERP."""
    merged = {}
    for key in set(sd0) | set(sd1):
        if key in sd0 and key in sd1:
            merged[key] = slerp(sd0[key], sd1[key], t=t)
        else:
            merged[key] = sd0.get(key, sd1.get(key))
    return merged


# ── TIES ──────────────────────────────────────────────────────────────────────

def ties_merge_adapters(state_dicts: list, lambda_: float = 0.2) -> dict:
    """TIES-Merging across a list of adapter state dicts.

    Steps:
      1. Trim  – zero out weights below lambda_ * max(|w|) per tensor.
      2. Elect – majority sign at each weight position.
      3. Merge – average only adapters whose sign agrees with elected sign.
    """
    all_keys = set()
    for sd in state_dicts:
        all_keys |= set(sd)

    merged = {}
    for key in all_keys:
        tensors = [sd[key].float() for sd in state_dicts if key in sd]
        if len(tensors) == 1:
            merged[key] = tensors[0].to(state_dicts[0][key].dtype)
            continue

        # Step 1 – Trim
        trimmed = []
        for t_w in tensors:
            thresh = lambda_ * t_w.abs().max()
            trimmed.append(torch.where(t_w.abs() < thresh, torch.zeros_like(t_w), t_w))

        stacked = torch.stack(trimmed)                        # [n, ...]
        # Step 2 – Elect sign
        elected_sign = torch.sign(stacked.sum(dim=0))
        # Step 3 – Disjoint merge
        agree_mask = (torch.sign(stacked) == elected_sign.unsqueeze(0)).float()
        n_agree = agree_mask.sum(dim=0).clamp(min=1)
        merged_val = (stacked * agree_mask).sum(dim=0) / n_agree

        src_dtype = next(sd[key].dtype for sd in state_dicts if key in sd)
        merged[key] = merged_val.to(src_dtype)

    return merged


# ── Helper: load all safetensors in a directory ───────────────────────────────

def _load_adapter_state_dict(adapter_dir: str) -> dict:
    sd = {}
    for fname in sorted(os.listdir(adapter_dir)):
        if fname.endswith(".safetensors"):
            sd.update(load_file(os.path.join(adapter_dir, fname)))
    return sd


# ── Run merge (TIES → SLERP polish) ──────────────────────────────────────────
# Add further checkpoint paths to ADAPTER_PATHS to enable ensemble merging.
ADAPTER_PATHS = [
    adapter_path,  # primary (LASER-pruned)
    # "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/19",  # example alternate
]
_MERGE_ADAPTER_PATH = "/kaggle/working/adapter-merged"

if len(ADAPTER_PATHS) >= 2:
    _state_dicts = [_load_adapter_state_dict(p) for p in ADAPTER_PATHS]

    # TIES for conflict resolution, then SLERP between TIES result and primary
    _ties_sd   = ties_merge_adapters(_state_dicts, lambda_=0.2)
    _merged_sd = slerp_merge_adapters(_state_dicts[0], _ties_sd, t=0.5)

    # Write merged weights: copy non-tensor files, overwrite safetensors
    if os.path.exists(_MERGE_ADAPTER_PATH):
        shutil.rmtree(_MERGE_ADAPTER_PATH)
    shutil.copytree(ADAPTER_PATHS[0], _MERGE_ADAPTER_PATH)
    save_file(_merged_sd, os.path.join(_MERGE_ADAPTER_PATH, "adapter_model.safetensors"))
    # Remove any other safetensors shards if they existed (merged into one file)
    for fname in os.listdir(_MERGE_ADAPTER_PATH):
        if fname.endswith(".safetensors") and fname != "adapter_model.safetensors":
            os.remove(os.path.join(_MERGE_ADAPTER_PATH, fname))

    adapter_path = _MERGE_ADAPTER_PATH
    print(f"Merge (TIES + SLERP) done → adapter_path = {adapter_path}")
else:
    print("Single checkpoint — skipping merge step.")


In [ ]:
import numpy as np

# ── Steering hyper-parameters ─────────────────────────────────────────────────
STEERING_ALPHA = 0.02   # fraction of steering vector added to each LoRA delta
STEERING_VECTORS_PATH = "/kaggle/input/steering-vectors/steering_vectors.pt"


def compute_steering_vector(
    pos_deltas: list,
    neg_deltas: list,
    use_pca: bool = False,
) -> torch.Tensor:
    """Compute a 'success direction' from lists of flattened delta tensors.

    Args:
        pos_deltas: 1-D float tensors from correct reasoning runs.
        neg_deltas: 1-D float tensors from incorrect reasoning runs.
        use_pca:    If True, use the first principal component of the
                    difference matrix instead of the raw mean difference.

    Returns:
        A 1-D steering tensor of the same dimension as each delta.
    """
    X_pos = torch.stack([d.float() for d in pos_deltas])   # [n_pos, D]
    X_neg = torch.stack([d.float() for d in neg_deltas])   # [n_neg, D]

    if use_pca:
        diff = torch.cat(
            [X_pos - X_pos.mean(0), -(X_neg - X_neg.mean(0))], dim=0
        )
        # Power-iteration for the first principal component
        v = diff[0].clone()
        v = v / v.norm()
        for _ in range(50):
            v = diff.T @ (diff @ v)
            v = v / v.norm()
        return v
    else:
        return X_pos.mean(0) - X_neg.mean(0)


def apply_steering_to_adapter(
    adapter_dir: str,
    steering_path: str,
    alpha: float = STEERING_ALPHA,
) -> None:
    """Inject alpha * steering_vector into each matching LoRA delta.

    Steering vectors are stored in a dict mapping a layer key prefix
    (e.g. 'base_model.model.model.layers.0.self_attn.q_proj') to a
    1-D tensor whose length equals out_dim * in_dim of that layer's delta.

    After injection the modified delta is re-factorised via truncated SVD
    back into (lora_B, lora_A) form, capped at LASER_MAX_RANK.
    """
    steering_vectors = torch.load(steering_path, map_location="cpu", weights_only=True)

    for fname in sorted(os.listdir(adapter_dir)):
        if not fname.endswith(".safetensors"):
            continue
        fpath = os.path.join(adapter_dir, fname)
        state_dict = load_file(fpath)

        lora_b_keys = [k for k in state_dict if "lora_B" in k and k.endswith(".weight")]
        changed = False
        for b_key in lora_b_keys:
            a_key = b_key.replace("lora_B", "lora_A")
            if a_key not in state_dict:
                continue

            layer_prefix = b_key.removesuffix(".lora_B.weight")
            if layer_prefix not in steering_vectors:
                continue

            B   = state_dict[b_key].float()
            A   = state_dict[a_key].float()
            delta = B @ A                                      # [out, in]

            sv = steering_vectors[layer_prefix].float()
            if sv.numel() != delta.numel():
                continue                                        # shape mismatch

            delta_steered = delta + alpha * sv.reshape(delta.shape)

            rank = min(B.shape[1], LASER_MAX_RANK)
            U, S, Vh = torch.linalg.svd(delta_steered, full_matrices=False)
            sroot = torch.sqrt(S[:rank])
            state_dict[b_key] = (U[:, :rank] * sroot.unsqueeze(0)).to(state_dict[b_key].dtype).contiguous()
            state_dict[a_key] = (sroot.unsqueeze(1) * Vh[:rank, :]).to(state_dict[a_key].dtype).contiguous()
            changed = True

        if changed:
            save_file(state_dict, fpath)
            print(f"  Steering applied: {fname}")


# ── Run steering (only if pre-computed vectors are available) ─────────────────
if os.path.isfile(STEERING_VECTORS_PATH):
    apply_steering_to_adapter(adapter_path, STEERING_VECTORS_PATH, alpha=STEERING_ALPHA)
    print(f"Representation steering done → adapter_path = {adapter_path}")
else:
    print(f"Steering vectors not found at {STEERING_VECTORS_PATH!r} — skipping.")


In [ ]:
from transformers import AutoTokenizer

# ── Numeric-bias hyper-parameters ─────────────────────────────────────────────
NUMERIC_BIAS_DELTA = 0.05   # additive boost in log-space for numeric token rows
_BASE_MODEL_PATH   = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# Surface forms to boost; the tokenizer will expand these to token IDs.
_NUMERIC_TOKEN_STRINGS = (
    [str(i) for i in range(10)]
    + ["AND", "OR", "XOR", "<<", ">>", "+", "-", "*", "/", "%", "=", "&", "|", "^", "~"]
)


def _get_numeric_token_ids(model_path: str) -> list:
    """Return a deduplicated sorted list of token IDs for numeric/logic tokens."""
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    ids = set()
    for tok in _NUMERIC_TOKEN_STRINGS:
        ids.update(tokenizer.encode(tok, add_special_tokens=False))
    return sorted(ids)


def adjust_numeric_bias(
    adapter_dir: str,
    model_path: str,
    delta: float = NUMERIC_BIAS_DELTA,
) -> None:
    """Boost lm_head lora_B rows for numeric/logic token IDs.

    If a lora_B tensor exists for the lm_head, the rows corresponding to
    numeric token IDs are incremented by `delta` (log-probability units).
    If no such tensor is found, the function reports and skips — adding a
    new lm_head LoRA from scratch would require matching rank metadata.
    """
    numeric_ids = _get_numeric_token_ids(model_path)
    print(f"  Numeric token IDs ({len(numeric_ids)}): {numeric_ids[:20]} ...")

    for fname in sorted(os.listdir(adapter_dir)):
        if not fname.endswith(".safetensors"):
            continue
        fpath = os.path.join(adapter_dir, fname)
        state_dict = load_file(fpath)

        lm_head_b_key = next(
            (k for k in state_dict if "lm_head" in k and "lora_B" in k and k.endswith(".weight")),
            None,
        )

        if lm_head_b_key is not None:
            W = state_dict[lm_head_b_key].float()
            # lora_B shape: [vocab_size, rank]
            valid_ids = [i for i in numeric_ids if i < W.shape[0]]
            W[valid_ids, :] += delta
            state_dict[lm_head_b_key] = W.to(state_dict[lm_head_b_key].dtype).contiguous()
            save_file(state_dict, fpath)
            print(f"  Numeric bias applied in {fname} (key: {lm_head_b_key}, boosted {len(valid_ids)} rows)")
        else:
            print(f"  No lm_head lora_B found in {fname} — skipping.")


# ── Run numeric bias adjustment ───────────────────────────────────────────────
adjust_numeric_bias(adapter_path, _BASE_MODEL_PATH, delta=NUMERIC_BIAS_DELTA)
print(f"Numeric bias adjustment done → adapter_path = {adapter_path}")


In [3]:
from tinker_cookbook import weights

weights.build_lora_adapter(
    base_model="/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    adapter_path=adapter_path,
    output_path="/kaggle/working/nemotron-adapter-ready-to-submit",
)


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


In [4]:
import shutil
shutil.make_archive('/kaggle/working/submission', 'zip', '/kaggle/working/nemotron-adapter-ready-to-submit')

'/kaggle/working/submission.zip'